# PsGameTranslator oyun tanıma eğitimi

Bu notebook, Drive'daki tamamlanmış IGDB veri setiyle `Llama-3.2-11B-Vision-Instruct` LoRA eğitir. A100 40 GB veya daha büyük Colab GPU kullan. Meta Llama lisansını Hugging Face'te kabul etmeden model indirilemez.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Drive'a TAM klasörü yükle: MyDrive/PsGameTranslator-IGDB-2016-2026
DATASET_ROOT = '/content/drive/MyDrive/PsGameTranslator-IGDB-2016-2026'
WORK_ROOT = '/content/drive/MyDrive/PsGameTranslatorTraining'
MODEL_ID = 'meta-llama/Llama-3.2-11B-Vision-Instruct'


In [ ]:
import json, os, torch
from pathlib import Path
report_path = Path(DATASET_ROOT) / 'metadata' / 'dataset_report.json'
assert report_path.is_file(), 'Drive a tam veri seti klasörünü yükle.'
report = json.loads(report_path.read_text())
print(report)
assert report['accepted_games'] >= 250
assert torch.cuda.is_available(), 'Colab GPU runtime seç.'
vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(torch.cuda.get_device_name(0), f'{vram:.1f} GB VRAM')
assert vram >= 40, 'Bu standart LoRA için A100 40 GB veya daha büyük GPU seç.'


In [ ]:
!pip -q install --upgrade "transformers==4.57.6" "datasets>=4.7,<5" "trl==1.8.0" "peft==0.19.1" "accelerate==1.14.0" safetensors pillow tensorboard
from huggingface_hub import login
from getpass import getpass
token = getpass('Yeni Hugging Face read token: ')
login(token=token)
os.environ['HF_TOKEN'] = token


In [ ]:
%%writefile /content/prepare_dataset.py
import argparse, json
from pathlib import Path
PROMPT = "You are looking at a screenshot from a video game. Look carefully at the HUD, UI style, art style, character models, and visible text or logos. What is the exact, real title of this specific game? Answer with ONLY the game's title. If you cannot identify it with confidence, answer Unknown."
def rows(path): return [json.loads(x) for x in path.read_text(encoding='utf-8').splitlines() if x.strip()]
def main():
 p=argparse.ArgumentParser(); p.add_argument('--root', required=True); a=p.parse_args(); root=Path(a.root); out=root/'training'/'llama32_vision'; out.mkdir(parents=True,exist_ok=True)
 for split in ('train','validation','test'):
  converted=[]
  for item in rows(root/'metadata'/f'{split}.jsonl'):
   image=(root/item['image']).resolve(); assert image.is_file(), image
   converted.append({'image':str(image),'game_id':item['game_id'],'title':item['title'],'messages':[{'role':'user','content':[{'type':'image'},{'type':'text','text':PROMPT}]},{'role':'assistant','content':[{'type':'text','text':item['title']}]}]})
  with (out/f'{split}.jsonl').open('w',encoding='utf-8') as f:
   for x in converted: f.write(json.dumps(x,ensure_ascii=False)+'\n')
  print(split,len(converted))
if __name__ == '__main__': main()

!python /content/prepare_dataset.py --root "$DATASET_ROOT"


In [ ]:
%%writefile /content/train_game_identifier.py
import argparse, json, torch
from pathlib import Path
from datasets import Image as DatasetImage, load_dataset
from peft import LoraConfig, TaskType
from transformers import AutoModelForImageTextToText, AutoProcessor
from trl import SFTConfig, SFTTrainer
p=argparse.ArgumentParser(); p.add_argument('--prepared',required=True); p.add_argument('--output',required=True); p.add_argument('--model',default='meta-llama/Llama-3.2-11B-Vision-Instruct'); a=p.parse_args()
def load(name): return load_dataset('json',data_files=str(Path(a.prepared)/f'{name}.jsonl'),split='train').cast_column('image',DatasetImage(decode=True))
processor=AutoProcessor.from_pretrained(a.model)
if processor.tokenizer.pad_token is None: processor.tokenizer.pad_token=processor.tokenizer.eos_token
model=AutoModelForImageTextToText.from_pretrained(a.model,torch_dtype=torch.bfloat16); model.config.use_cache=False
lora=LoraConfig(task_type=TaskType.CAUSAL_LM,r=32,lora_alpha=64,lora_dropout=.05,target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'])
cfg=SFTConfig(output_dir=a.output,num_train_epochs=4,per_device_train_batch_size=1,per_device_eval_batch_size=1,gradient_accumulation_steps=16,learning_rate=2e-5,lr_scheduler_type='cosine',warmup_ratio=.03,bf16=True,tf32=True,gradient_checkpointing=True,gradient_checkpointing_kwargs={'use_reentrant':False},optim='adamw_torch_fused',logging_steps=5,logging_first_step=True,eval_strategy='steps',eval_steps=100,save_strategy='steps',save_steps=100,save_total_limit=2,load_best_model_at_end=True,metric_for_best_model='eval_loss',greater_is_better=False,max_length=None,assistant_only_loss=True,remove_unused_columns=False,report_to=['tensorboard'])
trainer=SFTTrainer(model=model,args=cfg,train_dataset=load('train'),eval_dataset=load('validation'),processing_class=processor,peft_config=lora)
trainer.train(); metrics=trainer.evaluate(); adapter=Path(a.output)/'adapter'; trainer.save_model(str(adapter)); processor.save_pretrained(adapter); (Path(a.output)/'evaluation_metrics.json').write_text(json.dumps(metrics,indent=2)); print(metrics)


In [ ]:
PREPARED = f'{DATASET_ROOT}/training/llama32_vision'
OUTPUT = f'{WORK_ROOT}/llama-ps-game-recognizer'
!mkdir -p "$WORK_ROOT"
!accelerate launch --num_processes 1 --mixed_precision bf16 /content/train_game_identifier.py --prepared "$PREPARED" --output "$OUTPUT"


## Eğitim sonrası

Drive'da `PsGameTranslatorTraining/llama-ps-game-recognizer/adapter` oluşur. Bu klasörü Windows bilgisayarına indir ve aşağıdaki PowerShell komutunu çalıştır:

```powershell
.\create_ollama_model.ps1 -AdapterDir 'C:\...\adapter'
```

Sonra uygulamadaki `OllamaVisionModel` değeri: `llama-ps-game-recognizer:latest`.